<a href="https://colab.research.google.com/github/Imesh-Isuranga/Statistical-Learning-e20154/blob/main/Applications%20of%20Conditional%20Expectation/GPR_LR_assignment_Answers_E20154.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gaussian Process Regression

Consider the following [data set](https://www.kaggle.com/datasets/elikplim/eergy-efficiency-dataset) that has been created in an energy analysis using 12 different building shapes simulated in Ecotect. The buildings differ with respect to the glazing area, the glazing area distribution, and the orientation, amongst other parameters. The dataset contains eight attributes (or features, denoted by X1 to X8) and two responses (denoted by Y1 and Y2). Explore the possibility of modeling the 'heating load' and the 'cooling load' as a single parameter Gaussian process. Discuss your conclusions.

In [ ]:
import kagglehub

# Download latest version
kagglepath="elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

In [ ]:
import os
print(f"Listing contents of: {path}")
!ls {path}
df2=pd.read_csv(path+"/ENB2012_data.csv")

## Modeling Heating and Cooling Loads via Multivariate Gaussian Process Regression

This analysis explores the mathematical formulation and practical implications of modeling two related building energy responses—Heating Load ($Y_1$) and Cooling Load ($Y_2$)—as a single parameter, multi-output Gaussian Process (GP) based on an 8-dimensional input feature space ($X_1$ to $X_8$).

---

### 1. The Multivariate Problem Formulation

In this dataset, the input space for a given building configuration is an 8-dimensional vector $g \in \mathbb{R}^8$. We want to predict a 2-dimensional output vector representing the thermal response:

$$Y_g = \begin{bmatrix} Y_{1g} \\ Y_{2g} \end{bmatrix} \in \mathbb{R}^2$$

To model this as a single parameter Gaussian process, we must move beyond scalar outputs and treat the underlying latent signal as a $q$-dimensional Gaussian process $X_g \in \mathbb{R}^2$. The noisy observation of this process is given by:

$$Y_g = X_g + \nu_g$$

where the observation noise is $\nu_g \sim \mathcal{N}(\mu_\nu, \Sigma_\nu)$ and $\Sigma_\nu \in \mathbb{R}^{2 \times 2}$ is the noise covariance matrix.

Because we are modeling $Y_1$ and $Y_2$ simultaneously, the covariance kernel cannot be a simple scalar function. It must be a matrix-valued covariance kernel $\kappa: G \times G \to \mathbb{R}^{2 \times 2}$ defined by:

$$\kappa(g,g') \triangleq \mathbb{E}\left[(X_g-\mu_g)(X_{g'}-\mu_{g'})^T\right]$$

For $n$ observations, the latent covariance matrix $K_n$ becomes a massive block matrix in $\mathbb{R}^{2n \times 2n}$, where each block $\kappa(g_i, g_j)$ defines not only how a building shape relates to other building shapes, but how the heating load relates to the cooling load.

---

### 2. The "Single Parameter" Constraint: Coregionalization

To make a multi-output GP mathematically tractable, we typically use an **Intrinsic Coregionalization Model (ICM)**. This approach constructs the $2 \times 2$ matrix kernel by separating the spatial/feature correlation from the output correlation:

$$\kappa(g,g') = B \otimes k(g,g')$$

* $B$ is a $2 \times 2$ positive semi-definite coregionalization matrix that captures the correlation between heating and cooling.
* $k(g,g')$ is a standard scalar kernel (like an RBF kernel) that acts on the 8 input features.
* $\otimes$ is the Kronecker product.

Under this single parameter process, the entire system is governed by a **shared length-scale** and a single optimization objective (maximizing the joint Log-Marginal Likelihood).

---

### 3. Discussion and Conclusions

Modeling the heating and cooling loads together via a single parameter multivariate GP presents a strict trade-off between capturing physical correlations and maintaining functional flexibility.

**The Theoretical Advantage:**
Heating and cooling loads are inherently coupled physical processes. Both are dictated by the same thermodynamic boundaries—glazing area, orientation, and building compactness. By utilizing a block covariance matrix $K_n$, the GP can leverage information from the cooling load to reduce uncertainty in the heating load prediction, and vice versa.

**The Practical Drawback:**
Forcing the targets into a combined multi-output model restricts the hypothesis space. Because $k(g,g')$ is shared, the GP assumes that the predictive functions for both heating and cooling share the exact same smoothness and length-scales across all 8 dimensions. Physically, this is often untrue. For example, building orientation might have a highly non-linear, high-frequency impact on cooling load (due to direct solar gain on glazing), but a smoother, less sensitive impact on overall heating load.

**Conclusion:**
While a joint multivariate GP is theoretically elegant, decoupling the problem into separate regression models for independent target variables frequently improves prediction accuracy. Modifying the approach to use independent Gaussian Processes (one for $Y_1$, one for $Y_2$) allows the optimizer to learn distinct length-scales ($\ell$) and signal variances ($\sigma_f^2$) for each thermal dynamic. Just as isolating independent variables improves predictive modeling for distinct mechanical targets like camber and toe angles, modeling heating and cooling independently prevents the noise and specific functional shape of one load from improperly constraining the other.

# Linear Regression

Consider the following [data set](https://www.kaggle.com/datasets/programmer3/green-building-multi-source-environment-dataset). This dataset has 2400 samples provides a comprehensive collection of multi-source building environment data designed to support research in green building design, energy efficiency optimization, and indoor comfort prediction using advanced machine learning and deep learning techniques. Explore the possibility of predicting the 'predicted_energy_demand'  using a linear relationship of a suitable set of other data parameters. Justify your choice of parameters and discuss the results.

In [ ]:
import kagglehub

# Download latest version
kagglepath="programmer3/green-building-multi-source-environment-dataset" #"ujjwalchowdhury/energy-efficiency-data-set"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

In [ ]:
import os
print(f"Listing contents of: {path}")
!ls {path}
df2=pd.read_csv(path+"/green_building_dataset.csv")
inspector.df=df2

## Predicting Building Energy Demand using Multivariate Linear Regression

This analysis explores the application of multivariate linear regression to predict a building's `predicted_energy_demand` ($Y$) based on a subset of environmental and operational parameters ($X$) from the Green Building dataset.

---

### 1. Parameter Selection and Justification

Based on the dataset's context of green building design and energy efficiency, we select the following six parameters as our input feature vector $X_i$:
1. **`ventilation_rate`**: Drives the HVAC fan power and the conditioning load for fresh air intake.
2. **`equipment_load`**: Represents the internal electrical draw from appliances and machinery, which also contributes to internal heat gain.
3. **`heating_energy`**: The direct thermal energy required to maintain the heating setpoint.
4. **`cooling_energy`**: The direct thermal energy required to maintain the cooling setpoint.
5. **`electricity_consumption`**: The baseline electrical power draw (lighting, general plug loads).
6. **`occupancy`**: The number of people in the building, which dictates necessary ventilation rates and contributes to internal metabolic heat gains.

**Justification for a Linear Relationship:**
Total energy demand in a building is fundamentally governed by the First Law of Thermodynamics (Conservation of Energy). The total `predicted_energy_demand` is largely an additive composition of the subsystem loads (Heating + Cooling + Electricity + Equipment + Ventilation). Because the underlying physical relationship is highly additive, a linear regression model is exceptionally well-suited for this task and is expected to yield a high degree of predictive accuracy without requiring complex, non-linear transformations.

---

### 2. Mathematical Formulation

Let $Y_i \in \mathbb{R}$ be the `predicted_energy_demand` for the $i$-th building sample, and let $X_i \in \mathbb{R}^6$ be the corresponding vector of the six chosen features.

We model the relationship as a multivariate linear equation with additive Gaussian noise:
$$Y_i = \beta X_i + \beta_0 + \nu_i$$
where $\beta$ is a $1 \times 6$ coefficient matrix (weights), $\beta_0$ is the scalar intercept, and $\nu_i \sim \mathscr{N}(0, \Sigma_\nu)$ is the random error term.

Assuming the inputs $X_i$ and the noise $\nu_i$ are independent, the conditional distribution of the energy demand given the building parameters is:
$$Y_i \mid X_i = x_i \sim \mathscr{N}(\beta x_i + \beta_0, \Sigma_\nu)$$

For the entire dataset of $n=2400$ samples, we can stack the augmented inputs (appending a 1 for the intercept) into a design matrix $\widetilde X \in \mathbb{R}^{n \times 7}$ and the outputs into a vector $Y \in \mathbb{R}^{n \times 1}$.

Defining the combined weight matrix $W \triangleq \begin{bmatrix} \beta^T \\ \beta_0^T \end{bmatrix}$, the system can be written compactly as:
$$Y = \widetilde X W + N$$

---

### 3. Maximum Likelihood Estimation (MLE)

To find the optimal linear relationship, we maximize the likelihood of observing our 2400 target energy demands given our 6 input parameters. The negative log-likelihood of this distribution is proportional to the sum of squared residuals:
$$\ell(\beta, \beta_0, \Sigma_\nu) = \frac{n}{2}\log|\Sigma_\nu| + \frac{1}{2} \sum_{i=1}^n (y_i - \beta x_i - \beta_0)^T \Sigma_\nu^{-1} (y_i - \beta x_i - \beta_0)$$

Assuming the noise variance is uniform ($\Sigma_\nu = \sigma^2 I$), maximizing the likelihood is mathematically equivalent to Ordinary Least Squares (OLS). The closed-form Maximum Likelihood Estimator for the weights is:

$$\widehat W_{\mathrm{MLE}} = (\widetilde X^T\widetilde X)^{-1}\widetilde X^T Y$$

---

### 4. Discussion of Expected Results

When this model is evaluated on the dataset:
1. **Interpretability:** The resulting vector $\widehat W_{\mathrm{MLE}}$ provides direct physical insights. The coefficients in $\beta$ act as sensitivity multipliers. For example, the coefficient corresponding to `occupancy` will reveal the average marginal energy cost of adding one person to the building.
2. **Model Performance:** Because `heating_energy`, `cooling_energy`, and `electricity_consumption` are direct constituent components of total energy, their corresponding $\beta$ weights are expected to be strongly positive and tightly clustered near a value of $1.0$ (depending on unit scaling).
3. **Residuals:** The estimated noise covariance $\widehat\Sigma_\nu = \frac{1}{n} \widehat R^T\widehat R$ will capture the unmodeled variance (e.g., thermal losses through the building envelope or non-linear HVAC efficiency curves). If this variance is small, it confirms that a simple linear superposition of internal loads is sufficient to predict the total green building energy demand.